# GISLR · Stage 1 — architecture training

**Pipeline stage** for GISLR: shared feature caches → one training run per (architecture,
subset) → registry run folders → canonical-eval handoff. Replaces the four
`gislr.1.model.{gru,lstm,bilstm,cnn1d}.ipynb` notebooks, which differed only in an
architecture-identity block.

**Why one notebook.** "All else identical" is the entire premise of both the architecture
comparison (TODO §4) and the subset ablations (TODO §3.1) — a run whose batch size or LR
quietly differs from its comparators is not a benchmark, it is noise. Four files each carrying
their own `HYP` dict made that guarantee a manual chore, and the drift showed up in the docs
before it showed up in the code (README described batch 512 / lr 1e-3 long after the notebooks
moved to 1024 / 1.414e-3).

**Hyperparameters live in a file, not in a cell.**
[`src/config/gislr.training.json`](config/gislr.training.json) is the single source of truth,
read at run time by `modules.model.config`. Every architecture inherits the `shared` block;
an architecture can deviate only by naming a key in its own `overrides`, which §2 prints
alongside the resolved values — so a deviation is visible rather than something you diff four
files to find. Overriding a key that isn't in `shared` is a hard error, so a typo can't
masquerade as a tuned setting.

**Artifacts produced**

| artifact | path |
|---|---|
| per-subset feature caches (shared by every architecture) | `data/cache/gislr/features/{train,val}_<tag>_{data,offsets}.npy` |
| run-dir pointer per (arch, subset) — drives auto-resume | `data/cache/runs/gislr_<arch>_<tag>.txt` |
| registry run folder, one per training start | `data/models/<run_id>/` (run_id = epoch seconds) |
| — the run record (schema: README.md § "meta.json schema") | `data/models/<run_id>/meta.json` |
| — checkpoints (gitignored) | `data/models/<run_id>/{best,last}.pt` |
| — learning curves, per-epoch history, landmark indices | `data/models/<run_id>/assets/` |
| queryable run index (rebuild after evals) | `data/models/index.csv` |

**Resumability** — cache builds skip existing files and write atomically; training auto-resumes
an *interrupted* run from `last.pt` (checkpoint + `meta.json` + `assets/history.json` written
every epoch, early-stopping counter and LR-scheduler state included) via its pointer file. A
**finished** run is never reused: re-running an architecture section starts a fresh
epoch-seconds run folder per subset. An interrupt never forces a full rerun.

**Cell independence** — every section re-reads the config from disk and resolves its runs
through the registry's pointer files, so any single architecture section can be re-run on its
own after §1, with no dependency on the others having run.

**Design decisions**

- All reusable logic is in `modules/model/` — `architectures.py` is the single model-class
  definition, shared with `modules/scripts/eval_gru.py`, so state_dicts can never drift.
- Training reports through **one progress bar per run** (batch progress, losses, LR, plateau
  counter in the same bar) — no nested bars, no per-epoch print spam.
- Feature caches are built **once** in §3 and reused by all four architectures; they depend on
  (subset, coords) only, never on the model.
- **Training only.** Evaluation, export and Kaggle submission are stage 2, in
  [`gislr.2.models.evaluation.ipynb`](gislr.2.models.evaluation.ipynb).

**Kernel requirements**: project `.venv`, CWD = `src/`, CUDA GPU.

## 1. Setup

Imports and paths only — **no hyperparameters here**. They come from
`src/config/gislr.training.json` via `modules.model.config.load_config()`, which every later
section calls itself.

In [ ]:
# ============================================================
# Setup — imports and resolved paths (hyperparameters come from the config)
# ============================================================
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from modules.dataset.landmark.subsets import get_subset
from modules.model import (
    ARCHS,
    comparison_row,
    eval_command,
    load_config,
    load_meta,
    pointer_run_dir,
    save_learning_curves,
    subset_tag,
    train_from_config,
)
from modules.paths import CACHE_DIR, MODELS_DIR, gislr_dir

# download only the GISLR competition data (never resolve_datasets() here —
# that would pull every dataset incl. POPSIGN raw video)
DATA_DIR = gislr_dir()


def ptr_key(arch: str, subset: str, coords: str) -> str:
    """Pointer-file key for one (arch, subset, coords) run — the registry's
    auto-resume handle, also used by the report/eval sections to find runs."""
    return f"gislr_{arch}_{subset_tag(subset, coords)}"


print(f"DATA_DIR   = {DATA_DIR}")
print(f"CACHE_DIR  = {CACHE_DIR}")
print(f"MODELS_DIR = {MODELS_DIR}")
print(f"architectures available: {', '.join(ARCHS)}")

DATA_DIR   = C:\Users\user2\.cache\kagglehub\competitions\asl-signs
CACHE_DIR  = C:\Users\user2\sign2speech\src\data\cache
MODELS_DIR = C:\Users\user2\sign2speech\src\data\models
architectures available: gru, lstm, bilstm, cnn1d


## 2. Training configuration — the single source of truth

The whole grid, resolved, before anything trains. `overrides` is the column to watch: `—` means
the architecture uses the shared values exactly, which is what makes the comparison a
comparison. Anything else is a deliberate deviation and will show up in that run's `meta.json`.

To change a hyperparameter, edit [`src/config/gislr.training.json`](config/gislr.training.json)
and re-run this cell — no notebook cell holds these values.

In [ ]:
# ============================================================
# Load + display the training config (edit src/config/gislr.training.json)
# ============================================================
cfg = load_config()

print(f"config : {cfg.path}")
print(f"dataset: {cfg.dataset} · regime: {cfg.regime} · coords: {cfg.coords}")
print(f"source recorded in meta.json: {cfg.source}\n")

summary = pd.DataFrame(cfg.summary())
with pd.option_context("display.width", 220, "display.max_columns", None):
    display(summary)

deviating = summary[summary["overrides"] != "—"]
if len(deviating):
    print("NOTE: these architectures deviate from the shared hyperparameters —")
    print("      their runs are not all-else-identical comparators:")
    display(deviating[["architecture", "overrides"]])
else:
    print("all architectures use the shared hyperparameters — runs are directly comparable")

planned = [(a, s) for a in cfg.architectures if cfg.enabled(a) for s in cfg.subsets_for(a)]
print(f"\nplanned: {len(planned)} runs = "
      + " + ".join(f"{a}×{len(cfg.subsets_for(a))}"
                   for a in cfg.architectures if cfg.enabled(a)))

config : C:\Users\user2\sign2speech\src\config\gislr.training.json
dataset: gislr · regime: v2-plateau-300 · coords: xy
source recorded in meta.json: gislr.1.models.training.ipynb



,architecture,enabled,coords,subsets,overrides,batch_size,lr,hidden_size,num_layers,dropout,weight_decay,epochs,grad_clip,lr_factor,lr_patience,es_patience,es_min_delta
0,gru,True,xy,"ME_126, ME_132, FP_118",—,1024,0.001414,256,2,0.3,0.0001,300,5.0,0.5,5,15,0.001
1,lstm,True,xy,"ME_126, ME_132, FP_118",—,1024,0.001414,256,2,0.3,0.0001,300,5.0,0.5,5,15,0.001
2,bilstm,True,xy,"ME_126, ME_132, FP_118",—,1024,0.001414,256,2,0.3,0.0001,300,5.0,0.5,5,15,0.001
3,cnn1d,True,xy,"ME_126, ME_132, FP_118",num_layers=5,1024,0.001414,256,5,0.3,0.0001,300,5.0,0.5,5,15,0.001


NOTE: these architectures deviate from the shared hyperparameters —
      their runs are not all-else-identical comparators:


,architecture,overrides
3,cnn1d,num_layers=5



planned: 12 runs = gru×3 + lstm×3 + bilstm×3 + cnn1d×3


## 3. Canonical split check

Sanity-check the stratified 90/10 seed-42 split that every leaderboard run and
`modules/scripts/eval_gru.py` share (85,029 train / 9,448 val). Purely diagnostic — the training
driver calls `get_canonical_split()` itself.

In [ ]:
# ============================================================
# Canonical split sanity check — must match the leaderboard split exactly
# (the split itself is asserted inside modules.model.data.get_canonical_split)
# ============================================================
from modules.model import get_canonical_split, load_label_map

sign2idx = load_label_map(DATA_DIR)
train_split, val_split = get_canonical_split(DATA_DIR, sign2idx)
print(f"train {len(train_split)} / val {len(val_split)} · "
      f"{train_split['sign'].nunique()} classes · "
      f"val classes {val_split['sign'].nunique()}")
display(train_split.head(3))

train 85029 / val 9448 · 250 classes · val classes 250


,path,participant_id,sequence_id,sign,label
0,train_landmark_files/53618/4154717427.parquet,53618,4154717427,sad,189
1,train_landmark_files/37055/8607903.parquet,37055,8607903,duck,67
2,train_landmark_files/27610/3996084463.parquet,27610,3996084463,mad,138


## 4. Feature caches — built once, shared by every architecture

One flat float32 array + offsets index per (split, subset, coords) under
`data/cache/gislr/features/`, decoded from raw parquet once. Caches depend on the *data*, never
on the model, so all four architectures reuse them — this is the section that used to be
duplicated across four notebooks and rebuilt per notebook.

Skip-if-exists and atomic; delete a file pair to force a rebuild.

In [ ]:
# ============================================================
# Build the feature caches every architecture will need
# (covers every (subset, coords) pair in the config)
# Safe to re-run — existing caches are skipped.
# ============================================================
from modules.model import build_subset_cache, get_canonical_split, load_label_map
from modules.model.data import FEATURES_DIR

cfg = load_config()
sign2idx = load_label_map(DATA_DIR)
train_split, val_split = get_canonical_split(DATA_DIR, sign2idx)

wanted = {(s, cfg.coords_for(a)) for a in cfg.architectures if cfg.enabled(a)
          for s in cfg.subsets_for(a)}
print(f"{len(wanted)} (subset, coords) cache pair(s) needed: "
      + ", ".join(f"{s}/{c}" for s, c in sorted(wanted)))

for name, coords in sorted(wanted):
    subset = get_subset(name)
    build_subset_cache(train_split, "train", subset, coords, DATA_DIR)
    build_subset_cache(val_split, "val", subset, coords, DATA_DIR)

total_gb = sum(f.stat().st_size for f in FEATURES_DIR.glob("*_data.npy")) / 1e9
print(f"\ntotal feature-cache size on disk: {total_gb:.1f} GB")

3 (subset, coords) cache pair(s) needed: FP_118/xy, ME_126/xy, ME_132/xy

total feature-cache size on disk: 27.0 GB


## 5. `StreamingGRU`

Unidirectional (causal) GRU — **the deployment
architecture**. Everything else in this notebook exists to be measured against it: the LSTM as a
cell-for-cell alternative, the BiLSTM as an upper bound it can never reach, the CNN as a
different inductive bias entirely.

Trains one run per subset in the config. **Re-runnable on its own**: this cell reloads the
config from disk and needs nothing from the other architecture sections. Finished runs are never
reused — re-running starts a fresh run folder per subset.

In [5]:
# ============================================================
# Train gru — hyperparameters from src/config/gislr.training.json
# (tunable here: SUBSETS override for a one-off; None = whatever the config says)
# ============================================================
ARCH = "gru"
SUBSETS = None   # e.g. ["ME_126"] to train just one, without editing the config

run_dirs_gru = train_from_config(ARCH, subsets=SUBSETS)

for name, rd in run_dirs_gru.items():
    print(f"  {name}: {rd.name}")

gru · regime v2-plateau-300 · coords xy · subsets ['ME_126', 'ME_132', 'FP_118']
  hyperparameters from gislr.training.json (shared, no overrides)


gislr/gru/me126-xy · run 1784453891:   0%|          | 0/300 [00:00<?, ?it/s]

me126-xy: EARLY STOP at epoch 81 — no val-acc gain > 0.001 for 15 epochs
me126-xy: DONE best_val_acc=0.7566 run_dir=C:\Users\user2\sign2speech\src\data\models\1784453891


gislr/gru/me132-xy · run 1784454580:   0%|          | 0/300 [00:00<?, ?it/s]

me132-xy: EARLY STOP at epoch 85 — no val-acc gain > 0.001 for 15 epochs
me132-xy: DONE best_val_acc=0.7495 run_dir=C:\Users\user2\sign2speech\src\data\models\1784454580


gislr/gru/fp118-xy · run 1784455294:   0%|          | 0/300 [00:00<?, ?it/s]

fp118-xy: EARLY STOP at epoch 85 — no val-acc gain > 0.001 for 15 epochs
fp118-xy: DONE best_val_acc=0.7506 run_dir=C:\Users\user2\sign2speech\src\data\models\1784455294
  ME_126: 1784453891
  ME_132: 1784454580
  FP_118: 1784455294


## 6. `StreamingLSTM`

Unidirectional (causal) LSTM — streaming-viable, and the
direct LSTM-vs-GRU comparison (TODO §4): identical hidden size, layers, readout and head, so the
recurrent cell is the only variable.

Trains one run per subset in the config. **Re-runnable on its own**: this cell reloads the
config from disk and needs nothing from the other architecture sections. Finished runs are never
reused — re-running starts a fresh run folder per subset.

In [6]:
# ============================================================
# Train lstm — hyperparameters from src/config/gislr.training.json
# (tunable here: SUBSETS override for a one-off; None = whatever the config says)
# ============================================================
ARCH = "lstm"
SUBSETS = None   # e.g. ["ME_126"] to train just one, without editing the config

run_dirs_lstm = train_from_config(ARCH, subsets=SUBSETS)

for name, rd in run_dirs_lstm.items():
    print(f"  {name}: {rd.name}")

lstm · regime v2-plateau-300 · coords xy · subsets ['ME_126', 'ME_132', 'FP_118']
  hyperparameters from gislr.training.json (shared, no overrides)


gislr/lstm/me126-xy · run 1784455964:   0%|          | 0/300 [00:00<?, ?it/s]

me126-xy: EARLY STOP at epoch 89 — no val-acc gain > 0.001 for 15 epochs
me126-xy: DONE best_val_acc=0.7460 run_dir=C:\Users\user2\sign2speech\src\data\models\1784455964


gislr/lstm/me132-xy · run 1784456692:   0%|          | 0/300 [00:00<?, ?it/s]

me132-xy: EARLY STOP at epoch 89 — no val-acc gain > 0.001 for 15 epochs
me132-xy: DONE best_val_acc=0.7451 run_dir=C:\Users\user2\sign2speech\src\data\models\1784456692


gislr/lstm/fp118-xy · run 1784457430:   0%|          | 0/300 [00:00<?, ?it/s]

fp118-xy: EARLY STOP at epoch 80 — no val-acc gain > 0.001 for 15 epochs
fp118-xy: DONE best_val_acc=0.7356 run_dir=C:\Users\user2\sign2speech\src\data\models\1784457430
  ME_126: 1784455964
  ME_132: 1784456692
  FP_118: 1784457430


## 7. `BiLSTM`

Bidirectional LSTM — an **offline-only accuracy reference,
never a deployment candidate**. Its backward pass reads future frames, which the streaming
deployment target forbids (README § Constraints). It exists to price how much accuracy causality
costs: the gap between it and `StreamingLSTM` is that price, since the two differ only in
direction.

Trains one run per subset in the config. **Re-runnable on its own**: this cell reloads the
config from disk and needs nothing from the other architecture sections. Finished runs are never
reused — re-running starts a fresh run folder per subset.

In [ ]:
# ============================================================
# Train bilstm — hyperparameters from src/config/gislr.training.json
# (tunable here: SUBSETS override for a one-off; None = whatever the config says)
# ============================================================
ARCH = "bilstm"
SUBSETS = None   # e.g. ["ME_126"] to train just one, without editing the config

run_dirs_bilstm = train_from_config(ARCH, subsets=SUBSETS)

for name, rd in run_dirs_bilstm.items():
    print(f"  {name}: {rd.name}")

bilstm · regime v2-plateau-300 · coords xy · subsets ['ME_126', 'ME_132', 'FP_118']
  hyperparameters from gislr.training.json (shared, no overrides)


gislr/bilstm/me126-xy · run 1784447175:   4%|4         | 12/300 [00:00<?, ?it/s]

me126-xy: resumed at epoch 12, best 0.6581, plateau 1/15
me126-xy: EARLY STOP at epoch 84 — no val-acc gain > 0.001 for 15 epochs
me126-xy: DONE best_val_acc=0.7569 run_dir=C:\Users\user2\sign2speech\src\data\models\1784447175


gislr/bilstm/me132-xy · run 1784459026:   0%|          | 0/300 [00:00<?, ?it/s]

me132-xy: EARLY STOP at epoch 59 — no val-acc gain > 0.001 for 15 epochs
me132-xy: DONE best_val_acc=0.7476 run_dir=C:\Users\user2\sign2speech\src\data\models\1784459026


gislr/bilstm/fp118-xy · run 1784459817:   0%|          | 0/300 [00:00<?, ?it/s]

## 8. `CausalConv1D`

Dilated causal Conv1d stack — streaming-viable, and the
first step toward the 1st-place 1D-CNN + Transformer port. Every conv is left-padded so frame *t*
never sees *t+1*, and normalization is per-frame LayerNorm rather than BatchNorm (whose
statistics would mix future frames into past ones).

**Note**: this architecture currently scores ~0.54 against ~0.75 for the recurrent models — a
~20-point gap that is an architecture/hyperparameter problem specific to it, not the shared
~73% plateau (TODO §7).

Trains one run per subset in the config. **Re-runnable on its own**: this cell reloads the
config from disk and needs nothing from the other architecture sections. Finished runs are never
reused — re-running starts a fresh run folder per subset.

In [ ]:
# ============================================================
# Train cnn1d — hyperparameters from src/config/gislr.training.json
# (tunable here: SUBSETS override for a one-off; None = whatever the config says)
# ============================================================
ARCH = "cnn1d"
SUBSETS = None   # e.g. ["ME_126"] to train just one, without editing the config

run_dirs_cnn1d = train_from_config(ARCH, subsets=SUBSETS)

for name, rd in run_dirs_cnn1d.items():
    print(f"  {name}: {rd.name}")

## 9. Comparison across architectures

Reads every run **from disk** via the registry pointer files, so it works in a fresh kernel
without re-running any training section, and reports whatever exists so far.

`train_val_acc` here is the training loop's own best val accuracy — provisional. Leaderboard
comparisons require the canonical per-class eval (§10), which is also what
`gislr.2.models.evaluation.ipynb` builds its confusion matrices from.

In [ ]:
# ============================================================
# Learning curves + cross-architecture comparison table
# (tunables: REPORT_ARCHS, SHOW_CURVES) — loads runs via pointer files
# ============================================================
REPORT_ARCHS = None    # None = every architecture in the config
SHOW_CURVES = True     # per-run learning-curve figures (saved into each run's assets/)

cfg = load_config()
archs = REPORT_ARCHS or cfg.architectures

rows = []
for arch in archs:
    coords = cfg.coords_for(arch)
    for name in cfg.subsets_for(arch):
        rd = pointer_run_dir(ptr_key(arch, name, coords))
        if rd is None:
            continue
        if SHOW_CURVES:
            save_learning_curves(rd, title=f"{arch} · {name}/{coords} · run {rd.name}")
            plt.show()
        rows.append({"architecture": arch, **comparison_row(rd)})

if rows:
    table = (pd.DataFrame(rows)
             .sort_values("train_val_acc", ascending=False)
             .reset_index(drop=True))
    with pd.option_context("display.width", 220, "display.max_columns", None):
        display(table)

    # an interrupted run is not a result — exclude it from "best" rather than
    # letting a partially-trained model look like a bad architecture
    unfinished = table[~table["finished"]]
    if len(unfinished):
        print("INTERRUPTED runs (still training or stopped early by hand) — "
              "not comparable, excluded from the summary below:")
        display(unfinished[["architecture", "subset", "run_id", "epochs",
                            "train_val_acc"]])

    done = table[table["finished"]]
    if len(done):
        print("\nbest FINISHED run per architecture:")
        display(done.sort_values("train_val_acc", ascending=False)
                .groupby("architecture", as_index=False).first()
                .sort_values("train_val_acc", ascending=False)
                [["architecture", "subset", "coords", "n_params",
                  "train_val_acc", "eval_status"]])
else:
    print("no runs yet — train an architecture in §5-§8 first")

## 10. Canonical per-class eval handoff

Prints the exact `modules/scripts/eval_gru.py` command per run — **run those yourself**; this
notebook never evaluates. The eval promotes a run's `meta.json` to `eval_status: "canonical"` and
writes `assets/val_predictions.npz` (the input to every confusion matrix downstream).

**This is the last stage here.** Leaderboard, confusion matrices, TFLite export and Kaggle
submission all live in [`gislr.2.models.evaluation.ipynb`](gislr.2.models.evaluation.ipynb) —
whose §3 can also run this backfill in-process, if you'd rather not paste commands.

In [ ]:
# ============================================================
# Canonical per-class eval handoff (tunable: DOC_ARCHS)
# meta.json is already written every epoch by the training driver — this cell
# only prints the eval + index-rebuild commands for the user.
# ============================================================
DOC_ARCHS = None    # None = every architecture in the config

cfg = load_config()
archs = DOC_ARCHS or cfg.architectures

print("=== canonical per-class eval — run these from the repo root (user) ===")
n = 0
for arch in archs:
    coords = cfg.coords_for(arch)
    for name in cfg.subsets_for(arch):
        rd = pointer_run_dir(ptr_key(arch, name, coords))
        if rd is None:
            continue
        finished = load_meta(rd)["training"]["finished"]
        print(f"\n# {arch} · {name}/{coords} · run {rd.name}"
              + ("" if finished else "   (WAIT: training incomplete)"))
        print(eval_command(rd))
        n += 1

if n:
    print("\nafterwards, rebuild the index:")
    print(".venv/Scripts/python.exe src/modules/scripts/build_model_index.py")
    print("\nor run gislr.2.models.evaluation.ipynb §3, which backfills every "
          "pending run in-process.")
else:
    print("no runs yet — train an architecture in §5-§8 first")